# Multiplying and Dividing Monomials

Notebook 15 introduced the monomial — a single-term polynomial like $4x^2$, $-3x$, or $7$. Notebook 16 built all the exponent rules from scratch. This notebook combines both: applying those rules to monomials with coefficients and variables.

Every operation on monomials splits cleanly into two independent steps that happen in parallel. Once you see the split, every problem becomes mechanical.

**Prerequisites:** Variables and Equations (07), Polynomial Operations — Adding and Subtracting (15), Exponent Multiplication and Division (16).

---

## 1. Anatomy of a Monomial

A monomial has two parts that live independently:

$$\underbrace{4}_{\text{coefficient}} \cdot \underbrace{x^2}_{\text{variable part}}$$

The **coefficient** is a plain number — a scalar. The **variable part** is one or more variables raised to powers.

**Programmer bridge:** a monomial is a struct with two fields: `coeff` (a number) and `exp` (a non-negative integer). The fields follow completely different rules when you combine monomials, so keeping them separate in your mental model matters.

| Expression | Coefficient | Variable | Exponent | Python repr |
|-----------|-------------|----------|----------|-------------|
| $4x^2$ | $4$ | $x$ | $2$ | `(4, 2)` |
| $-3x^5$ | $-3$ | $x$ | $5$ | `(-3, 5)` |
| $7$ | $7$ | — | $0$ | `(7, 0)` |
| $x$ | $1$ | $x$ | $1$ | `(1, 1)` |
| $-x^3$ | $-1$ | $x$ | $3$ | `(-1, 3)` |

In [ ]:
def mono_str(coeff, exp, var='x'):
    """Format (coeff, exp) as a readable monomial string."""
    if exp == 0:
        return str(coeff)
    var_part = var if exp == 1 else f'{var}^{exp}'
    if coeff == 1:
        return var_part
    if coeff == -1:
        return f'-{var_part}'
    return f'{coeff}{var_part}'

def mono_eval(coeff, exp, x):
    return coeff * (x ** exp)

monomials = [(4, 2), (-3, 5), (7, 0), (1, 1), (-1, 3), (2, 7)]

print(f'  {"(coeff, exp)":<14}  {"String":<10}  value at x=2')
print(f'  {"-"*14}  {"-"*10}  ------------')
for c, e in monomials:
    print(f'  {str((c, e)):<14}  {mono_str(c, e):<10}  {mono_eval(c, e, 2)}')

---

## 2. Multiplying Monomials

What happens when you multiply two monomials?

$$3x^2 \times 4x^3$$

Multiplication is commutative and associative — you can regroup factors freely. Pull the numbers together and the $x$-copies together:

$$= (3 \times 4) \times (x^2 \times x^3)$$

The numbers multiply as plain arithmetic. The $x$ copies add by the product rule from notebook 16:

$$= 12 \times x^{2+3} = 12x^5$$

**The rule:** coefficients multiply, exponents add.

$$c_1 x^a \cdot c_2 x^b = (c_1 \cdot c_2)\, x^{a+b}$$

**Programmer bridge:** it is a parallel operation on two independent fields:
```python
(c1, e1) * (c2, e2)  =>  (c1 * c2,  e1 + e2)
                          ^^^plain    ^^^product rule
```
The coefficient path uses ordinary number multiplication. The exponent path uses the product rule. They never interfere with each other.

In [ ]:
def mono_multiply(c1, e1, c2, e2):
    """Multiply two monomials: coefficients multiply, exponents add."""
    return c1 * c2, e1 + e2

def show_multiply(c1, e1, c2, e2):
    c_res, e_res = mono_multiply(c1, e1, c2, e2)
    print(f'  {mono_str(c1, e1)} × {mono_str(c2, e2)}')
    print(f'  = ({c1} × {c2}) · x^({e1}+{e2})')
    print(f'  = {mono_str(c_res, e_res)}')
    x = 3
    orig   = mono_eval(c1, e1, x) * mono_eval(c2, e2, x)
    result = mono_eval(c_res, e_res, x)
    print(f'  Verify at x=3: {orig} = {result}  {"✓" if orig == result else "✗"}')
    print()

show_multiply(3,  2, 4,  3)   #  3x^2 ×  4x^3 = 12x^5
show_multiply(2,  4, 5,  2)   #  2x^4 ×  5x^2 = 10x^6
show_multiply(-3, 2, 4,  5)   # -3x^2 ×  4x^5 = -12x^7
show_multiply(1,  1, 1,  1)   #     x ×      x =  x^2
show_multiply(6,  3, 1,  0)   #  6x^3 ×      1 =  6x^3  (multiply by a constant)

---

## 3. Dividing Monomials

Division follows the same split:

$$\frac{12x^5}{4x^2}$$

Separate the coefficient fraction from the variable fraction:

$$= \frac{12}{4} \cdot \frac{x^5}{x^2}$$

The numbers divide as plain arithmetic. The $x$ copies cancel by the quotient rule from notebook 16:

$$= 3 \cdot x^{5-2} = 3x^3$$

**The rule:** coefficients divide, exponents subtract.

$$\frac{c_1 x^a}{c_2 x^b} = \frac{c_1}{c_2}\, x^{a-b}$$

**Programmer bridge:**
```python
(c1, e1) / (c2, e2)  =>  (c1 / c2,  e1 - e2)
                          ^^^plain    ^^^quotient rule
```

When $a < b$, the exponent result is negative. A negative exponent means the variable moves to the denominator — $x^{-3} = 1/x^3$.

In [ ]:
from fractions import Fraction

def mono_divide(c1, e1, c2, e2):
    """Divide two monomials: coefficients divide, exponents subtract."""
    return Fraction(c1, c2), e1 - e2

def show_divide(c1, e1, c2, e2):
    c_res, e_res = mono_divide(c1, e1, c2, e2)
    print(f'  {mono_str(c1, e1)} ÷ {mono_str(c2, e2)}')
    print(f'  = ({c1} ÷ {c2}) · x^({e1}-{e2})')
    c_display = int(c_res) if c_res.denominator == 1 else c_res
    if e_res < 0:
        print(f'  = {c_display}x^{e_res}  =  {c_display}/x^{abs(e_res)}')
    else:
        print(f'  = {mono_str(int(c_res) if c_res.denominator == 1 else c_res, e_res)}')
    x = 5
    orig   = Fraction(mono_eval(c1, e1, x), mono_eval(c2, e2, x))
    result = c_res * Fraction(x) ** e_res
    print(f'  Verify at x=5: {orig} = {result}  {"✓" if orig == result else "✗"}')
    print()

show_divide(12, 5,  4, 2)   # 12x^5 ÷ 4x^2  = 3x^3
show_divide(20, 7,  4, 3)   # 20x^7 ÷ 4x^3  = 5x^4
show_divide(15, 4, -3, 1)   # 15x^4 ÷ -3x   = -5x^3
show_divide( 6, 2,  9, 5)   #  6x^2 ÷ 9x^5  = (2/3)x^-3 = 2/(3x^3)

---

## 4. Raising a Monomial to a Power

What about $(3x^2)^4$?

The power of a product rule from notebook 16: $(ab)^n = a^n b^n$. The coefficient and the variable part are both factors, so the exponent distributes to both:

$$(3x^2)^4 = 3^4 \cdot (x^2)^4 = 81 \cdot x^{2 \times 4} = 81x^8$$

**The rule:** coefficient raised to the power, exponent multiplied by the power.

$$\left(c\,x^a\right)^n = c^n\, x^{a \cdot n}$$

**Programmer bridge:**
```python
(c, e) ** n  =>  (c**n,  e * n)
                  ^^^       ^^^power rule
```

> The coefficient gets raised too. A very common error is forgetting to apply the exponent to the coefficient. $(3x^2)^4 \neq 3x^8$ — the $3$ must be raised to the fourth power.

In [ ]:
def mono_power(coeff, exp, n):
    """Raise a monomial to a power: coefficient to the power, exponent multiplied."""
    return coeff ** n, exp * n

def show_power(coeff, exp, n):
    c_res, e_res = mono_power(coeff, exp, n)
    print(f'  ({mono_str(coeff, exp)})^{n}')
    print(f'  = {coeff}^{n} · x^({exp}×{n})')
    print(f'  = {mono_str(c_res, e_res)}')
    x = 2
    orig   = mono_eval(coeff, exp, x) ** n
    result = mono_eval(c_res, e_res, x)
    print(f'  Verify at x=2: {orig} = {result}  {"✓" if orig == result else "✗"}')
    print()

show_power(3,  2, 4)   # (3x^2)^4  = 81x^8
show_power(2,  3, 3)   # (2x^3)^3  = 8x^9
show_power(-2, 1, 3)   # (-2x)^3   = -8x^3
show_power(-2, 1, 2)   # (-2x)^2   = 4x^2  (even power — negative becomes positive)
show_power(5,  0, 3)   # 5^3       = 125   (no variable, just number)

---

## 5. Multi-Variable Monomials

Monomials can have more than one variable: $3x^2y$, $-5x^3y^2z$, $2ab^4$.

Each variable is tracked independently. When you multiply, divide, or raise to a power, the same rule applies to each variable separately.

**Example — multiply:**

$$2x^3y^2 \times 4x y^3$$

Group by type — coefficient, $x$ terms, $y$ terms:

$$= (2 \times 4) \cdot (x^3 \times x^1) \cdot (y^2 \times y^3) = 8x^4y^5$$

**Programmer bridge:** a multi-variable monomial is a coefficient with a dict of `{variable: exponent}`. Multiplying means multiplying coefficients and merging the dicts by adding values for matching keys — exactly like a `Counter` merge:

```python
# 2x^3y^2  *  4xy^3
coeff: 2 * 4 = 8
vars:  {'x': 3, 'y': 2}  +  {'x': 1, 'y': 3}  =  {'x': 4, 'y': 5}
```

Variables that only appear in one monomial just carry over unchanged — their exponent in the other monomial is implicitly 0.

In [ ]:
from fractions import Fraction

def mv_str(coeff, vars_dict):
    """Format a multi-variable monomial (coeff, {var: exp}) as a string."""
    if coeff == 0:
        return '0'
    parts = []
    if coeff == 1 and vars_dict:
        pass
    elif coeff == -1 and vars_dict:
        parts.append('-')
    else:
        parts.append(str(coeff))
    for var in sorted(vars_dict):
        exp = vars_dict[var]
        if exp == 0:
            continue
        parts.append(var if exp == 1 else f'{var}^{exp}')
    return ''.join(parts) if parts else '1'

def mv_eval(coeff, vars_dict, val_map):
    result = coeff
    for var, exp in vars_dict.items():
        result *= val_map[var] ** exp
    return result

def mv_multiply(c1, v1, c2, v2):
    """Multiply two multi-variable monomials."""
    c_res = c1 * c2
    all_vars = set(v1) | set(v2)
    v_res = {var: v1.get(var, 0) + v2.get(var, 0) for var in all_vars}
    v_res = {k: v for k, v in v_res.items() if v != 0}
    return c_res, v_res

def mv_divide(c1, v1, c2, v2):
    """Divide two multi-variable monomials."""
    c_res = Fraction(c1, c2)
    all_vars = set(v1) | set(v2)
    v_res = {var: v1.get(var, 0) - v2.get(var, 0) for var in all_vars}
    v_res = {k: v for k, v in v_res.items() if v != 0}
    return c_res, v_res

def mv_power(coeff, vars_dict, n):
    """Raise a multi-variable monomial to a power."""
    return coeff ** n, {var: exp * n for var, exp in vars_dict.items()}

vals = {'x': 2, 'y': 3, 'z': 5}

print('Multiply:')
cases = [
    ((2, {'x': 3, 'y': 2}), (4, {'x': 1, 'y': 3})),   # 2x^3y^2 * 4xy^3
    ((3, {'x': 2, 'y': 1}), (5, {'x': 1, 'y': 2})),   # 3x^2y * 5xy^2
    ((2, {'x': 3         }), (4, {         'y': 2})),   # 2x^3 * 4y^2 — different vars
]
for (c1, v1), (c2, v2) in cases:
    c_res, v_res = mv_multiply(c1, v1, c2, v2)
    c_display = int(c_res) if isinstance(c_res, Fraction) and c_res.denominator == 1 else c_res
    orig   = mv_eval(c1, v1, vals) * mv_eval(c2, v2, vals)
    result = mv_eval(c_display, v_res, vals)
    print(f'  {mv_str(c1, v1)} × {mv_str(c2, v2)} = {mv_str(c_display, v_res)}',
          f'  (verify: {orig} = {result} {"✓" if orig == result else "✗"})')

print()
print('Divide:')
div_cases = [
    ((12, {'x': 5, 'y': 3}), (4, {'x': 2, 'y': 1})),   # 12x^5y^3 / 4x^2y
    ((10, {'x': 3, 'y': 4}), (2, {'x': 1, 'y': 2})),   # 10x^3y^4 / 2xy^2
]
for (c1, v1), (c2, v2) in div_cases:
    c_res, v_res = mv_divide(c1, v1, c2, v2)
    c_display = int(c_res) if c_res.denominator == 1 else c_res
    orig   = Fraction(mv_eval(c1, v1, vals), mv_eval(c2, v2, vals))
    result = mv_eval(c_display, v_res, vals)
    print(f'  {mv_str(c1, v1)} ÷ {mv_str(c2, v2)} = {mv_str(c_display, v_res)}',
          f'  (verify: {orig} = {result} {"✓" if orig == result else "✗"})')

print()
print('Power:')
pow_cases = [
    (2, {'x': 2, 'y': 1}, 3),   # (2x^2y)^3 = 8x^6y^3
    (3, {'x': 1, 'y': 2}, 2),   # (3xy^2)^2 = 9x^2y^4
]
for coeff, vd, n in pow_cases:
    c_res, v_res = mv_power(coeff, vd, n)
    orig   = mv_eval(coeff, vd, vals) ** n
    result = mv_eval(c_res, v_res, vals)
    print(f'  ({mv_str(coeff, vd)})^{n} = {mv_str(c_res, v_res)}',
          f'  (verify: {orig} = {result} {"✓" if orig == result else "✗"})')

---

## 6. Worked Examples

Each example applies one or more operations in sequence. The key habit: **identify the operation type first** (multiply / divide / power), then handle coefficients and exponents as two separate tracks.

### Example 1 — Multiply, negative coefficient

$$(-3x^2)(4x^5)$$

Coefficients: $-3 \times 4 = -12$. Exponents: $2 + 5 = 7$.

$$= -12x^7$$

### Example 2 — Divide, result is a fraction

$$\frac{6x^3}{9x^5}$$

Coefficients: $\frac{6}{9} = \frac{2}{3}$. Exponents: $3 - 5 = -2$.

$$= \frac{2}{3}x^{-2} = \frac{2}{3x^2}$$

### Example 3 — Power of a monomial

$$(2x^3)^4$$

Coefficient: $2^4 = 16$. Exponent: $3 \times 4 = 12$.

$$= 16x^{12}$$

### Example 4 — Power then divide (chain)

$$\frac{(3x^2)^2}{6x^3}$$

**Step 1** — expand the power: $(3x^2)^2 = 9x^4$

**Step 2** — divide: $\frac{9x^4}{6x^3}$. Coefficients: $\frac{9}{6} = \frac{3}{2}$. Exponents: $4 - 3 = 1$.

$$= \frac{3}{2}x = \frac{3x}{2}$$

### Example 5 — Multiply three monomials

$$2x^3 \cdot 3x \cdot 5x^2$$

Multiply all coefficients: $2 \times 3 \times 5 = 30$. Add all exponents: $3 + 1 + 2 = 6$.

$$= 30x^6$$

### Example 6 — Multi-variable, power

$$(2x^2y)^3$$

Coefficient: $2^3 = 8$. Exponent of $x$: $2 \times 3 = 6$. Exponent of $y$: $1 \times 3 = 3$.

$$= 8x^6y^3$$

### Example 7 — Multi-variable, multiply then simplify

$$\frac{4x^3y^2 \cdot 3xy}{6x^2y^2}$$

**Step 1** — multiply numerator: $4x^3y^2 \cdot 3xy = 12x^4y^3$

**Step 2** — divide: $\frac{12x^4y^3}{6x^2y^2}$. Coefficient: $12 \div 6 = 2$. Exponents: $x: 4-2=2$, $y: 3-2=1$.

$$= 2x^2y$$

In [ ]:
from fractions import Fraction

x = 3
vals = {'x': x, 'y': 2}

examples = [
    ('Ex 1: (-3x^2)(4x^5) = -12x^7',
      mono_eval(-3, 2, x) * mono_eval(4, 5, x),
      mono_eval(-12, 7, x)),

    ('Ex 2: 6x^3 / 9x^5 = 2/(3x^2)',
      Fraction(mono_eval(6, 3, x), mono_eval(9, 5, x)),
      Fraction(2, 3) * Fraction(x) ** -2),

    ('Ex 3: (2x^3)^4 = 16x^12',
      mono_eval(2, 3, x) ** 4,
      mono_eval(16, 12, x)),

    ('Ex 4: (3x^2)^2 / 6x^3 = (3/2)x',
      Fraction(mono_eval(3, 2, x) ** 2, mono_eval(6, 3, x)),
      Fraction(3, 2) * x),

    ('Ex 5: 2x^3 · 3x · 5x^2 = 30x^6',
      mono_eval(2, 3, x) * mono_eval(3, 1, x) * mono_eval(5, 2, x),
      mono_eval(30, 6, x)),

    ('Ex 6: (2x^2y)^3 = 8x^6y^3',
      mv_eval(2, {'x': 2, 'y': 1}, vals) ** 3,
      mv_eval(8, {'x': 6, 'y': 3}, vals)),

    ('Ex 7: 4x^3y^2 · 3xy / 6x^2y^2 = 2x^2y',
      Fraction(mv_eval(4, {'x':3,'y':2}, vals) * mv_eval(3, {'x':1,'y':1}, vals),
               mv_eval(6, {'x':2,'y':2}, vals)),
      mv_eval(2, {'x': 2, 'y': 1}, vals)),
]

print(f'Verifying all examples at x={x}, y={vals["y"]}:')
print()
for label, computed, expected in examples:
    ok = computed == expected
    print(f'  {label}')
    print(f'    computed={computed},  expected={expected}  {"✓" if ok else "✗ MISMATCH"}')
    print()

---

## 7. Visualisation — Multiplying Functions

Multiplying two monomials produces a new monomial with a higher degree. The degree tells you how fast the function grows — a higher degree curves upward more steeply.

Below: $f(x) = 2x^2$ and $g(x) = 3x^3$ are plotted separately. Their product $h(x) = f(x) \cdot g(x) = 6x^5$ is plotted too. You can see the product curving away from both factors — a higher-degree polynomial grows faster than either of its constituent monomials.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

x = np.linspace(0, 2.5, 300)

# Two monomials and their product
c1, e1 = 2, 2    # f = 2x^2
c2, e2 = 3, 3    # g = 3x^3
c_prod, e_prod = c1 * c2, e1 + e2   # product rule: 6x^5

f = c1 * x**e1
g = c2 * x**e2
h = c_prod * x**e_prod

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Multiplying Monomials — what happens to the function', fontsize=13, fontweight='bold')

# Left: the three functions
ax = axes[0]
ax.plot(x, f, color='steelblue',  linewidth=2.5, label=f'f(x) = {mono_str(c1, e1)}')
ax.plot(x, g, color='tomato',     linewidth=2.5, label=f'g(x) = {mono_str(c2, e2)}')
ax.plot(x, h, color='seagreen',   linewidth=2.5, label=f'h(x) = f·g = {mono_str(c_prod, e_prod)}', linestyle='--')
ax.set_xlim(0, 2.5)
ax.set_ylim(0, 60)
ax.set_xlabel('x', fontsize=11)
ax.set_ylabel('y', fontsize=11)
ax.set_title('f, g, and their product h = f·g', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.annotate(
    f'Multiply:\ncoeff: {c1}×{c2}={c_prod}\nexp:   {e1}+{e2}={e_prod}',
    xy=(1.5, 25), fontsize=9,
    bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', edgecolor='grey', alpha=0.9)
)

# Right: degree comparison — show how exponents stack
ax2 = axes[1]
exps    = [e1, e2, e_prod]
labels  = [f'{mono_str(c1,e1)}\n(degree {e1})',
           f'{mono_str(c2,e2)}\n(degree {e2})',
           f'{mono_str(c_prod,e_prod)}\n(degree {e_prod})']
colors  = ['steelblue', 'tomato', 'seagreen']
bars    = ax2.bar(labels, exps, color=colors, alpha=0.8, edgecolor='white', linewidth=1.5)
for bar, exp in zip(bars, exps):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(exp), ha='center', va='bottom', fontsize=12, fontweight='bold')
ax2.set_ylabel('Degree (exponent)', fontsize=11)
ax2.set_title(f'Degree adds: {e1} + {e2} = {e_prod}', fontsize=11)
ax2.set_ylim(0, e_prod + 1.5)
ax2.axhline(e1 + e2, color='seagreen', linestyle=':', linewidth=1.5, alpha=0.7)
ax2.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---

## Exercises

Work each one by hand — identify the operation, handle coefficients separately from exponents — then verify in code.

1. Simplify $5x^3 \cdot 2x^4$.
2. Simplify $\dfrac{18x^7}{6x^3}$.
3. Simplify $(-4x^2)^3$. Does the negative sign survive? Why?
4. Simplify $\dfrac{15x^6}{-3x^2}$.
5. Simplify $\dfrac{x^5 \cdot x^3}{x^4}$. Use product rule first, then quotient rule.
6. Simplify $(5x^3)^2 \cdot 2x$. Power first, then multiply.
7. Simplify $\dfrac{(3x^2)^2}{9x^4}$. What do you expect the result to be, before computing?
8. Simplify $3x^2y \cdot 4xy^3$. Multi-variable.
9. Simplify $(2xy^2)^3$. Raise each variable's exponent separately.
10. Simplify $\dfrac{6x^3y^2}{9xy^4}$. Coefficient divides; each exponent subtracts independently.

In [ ]:
from fractions import Fraction

# Verify your answers here
# Use mono_eval(coeff, exp, x) for single-variable
# Use mv_eval(coeff, vars_dict, val_map) for multi-variable
x = 3
vals = {'x': x, 'y': 2}

# Example: exercise 1
# 5x^3 · 2x^4  ==  10x^7
# print(mono_eval(5, 3, x) * mono_eval(2, 4, x) == mono_eval(10, 7, x))

---

## Formal Notation

A **monomial** is an expression of the form $c\,x^n$ where $c \in \mathbb{R}$ is the coefficient and $n \in \mathbb{Z}_{\geq 0}$ is the degree.

For monomials $c_1 x^a$ and $c_2 x^b$:

| Operation | Formula | Tracks |
|-----------|---------|--------|
| **Multiply** | $c_1 x^a \cdot c_2 x^b = (c_1 c_2)\, x^{a+b}$ | coeff ×; exponent + |
| **Divide** | $\dfrac{c_1 x^a}{c_2 x^b} = \dfrac{c_1}{c_2}\, x^{a-b}$ | coeff ÷; exponent − |
| **Power** | $(c\,x^a)^n = c^n x^{an}$ | coeff ^; exponent × |

For a **multi-variable monomial** $c\,x^a y^b$:

$$c_1 x^{a_1} y^{b_1} \cdot c_2 x^{a_2} y^{b_2} = (c_1 c_2)\, x^{a_1+a_2} y^{b_1+b_2}$$

The same rules apply to each variable independently. Variables that appear in only one factor are carried unchanged (their exponent in the other factor is implicitly $0$).

These rules are direct consequences of the exponent rules established in notebook 16 — no new axioms are needed.

---

## Real-World Connection

- **Dimensional analysis:** physical units are multi-variable monomials. Velocity is $\text{m} \cdot \text{s}^{-1}$, time is $\text{s}^1$. Distance $= v \times t = \text{m} \cdot \text{s}^{-1} \cdot \text{s}^1 = \text{m} \cdot \text{s}^{0} = \text{m}$. Engineers combine units by applying the product rule — adding exponents on matching unit-variables. The exponents on $\text{kg}$, $\text{m}$, and $\text{s}$ each track independently, exactly like a multi-variable monomial.

- **Polynomial regression and feature engineering:** in machine learning, polynomial features are generated by multiplying monomials. If the raw features are $x$ and $y$, then the polynomial feature set $\{x, y, x^2, xy, y^2, x^3, \ldots\}$ is built by multiplying monomials together. The exponents you choose determine the interaction terms captured by the model.

- **Complexity analysis:** Big-O products use the product rule. An algorithm that loops $O(n^2)$ times and does $O(n^3)$ work per iteration costs $O(n^2) \cdot O(n^3) = O(n^5)$. The $n$ is the variable, the exponents add — you are multiplying monomials.

- **Computer graphics — homogeneous coordinates:** a 3D point $(x, y, z)$ in homogeneous coordinates is $(x, y, z, w)$. Scaling all coordinates by a factor $\lambda$ gives $(\lambda x, \lambda y, \lambda z, \lambda w)$, which is equivalent to scaling $w$ alone. This is the power rule on a monomial with a coefficient — the same math underlies every matrix transform in a 3D engine.

---

## Summary

Every operation on monomials splits into two parallel tracks — the coefficient track and the exponent track — that never interfere:

| Operation | Coefficient track | Exponent track |
|-----------|-------------------|----------------|
| Multiply | multiply ($\times$) | add ($+$) — product rule |
| Divide | divide ($\div$) | subtract ($-$) — quotient rule |
| Raise to power | raise ($\text{exp}$) | multiply ($\times$) — power rule |

The exponent rules from notebook 16 do all the work on the variable side. The coefficient side is plain arithmetic. Keep the two tracks separate and the operations become mechanical.

Multi-variable monomials extend the same idea: each variable has its own exponent track. Apply the rule per variable independently — like a dict merge where you add (or subtract, or multiply) the values for each key separately.

When a result has a negative exponent, it means the variable moved to the denominator: $x^{-n} = 1/x^n$.